In [ ]:
import os
import numpy as np
import tiktoken
from tqdm import tqdm
from datasets import load_dataset

In [2]:
os.chdir('../')

In [3]:
num_proc = os.cpu_count()
print(num_proc)

11


In [4]:
dataset = load_dataset("roneneldan/TinyStories")

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

In [6]:
enc = tiktoken.get_encoding("gpt2")

In [7]:
enc

<Encoding 'gpt2'>

In [8]:
eot = enc._special_tokens["<|endoftext|>"]
print(eot)

50256


In [9]:
def tokenize(row):
    tokens = enc.encode_ordinary(row["text"]) + [eot]
    length = len(tokens)
    return {"tokens": tokens, "length": length}

In [10]:
train_ds = dataset["train"]
val_ds = dataset["validation"]

In [11]:
print("Length of train_ds:", len(train_ds))
print("Length of val_ds:", len(val_ds))

Length of train_ds: 2119719
Length of val_ds: 21990


In [12]:
train_tokenized_dataset = train_ds.map(
    tokenize,
    remove_columns=["text"],
    num_proc=num_proc,
    desc="Tokenizing training dataset",
)

In [13]:
val_tokenized_dataset = val_ds.map(
    tokenize,
    remove_columns=["text"],
    num_proc=num_proc,
    desc="Tokenizing validation dataset",
)

In [14]:
train_tokenized_dataset.to_pandas().head(5)

,tokens,length
0,"[3198, 1110, 11, 257, 1310, 2576, 3706, 20037,...",163
1,"[7454, 2402, 257, 640, 11, 612, 373, 257, 1310...",178
2,"[3198, 1110, 11, 257, 1310, 5916, 3706, 4463, ...",213
3,"[7454, 2402, 257, 640, 11, 287, 257, 1956, 133...",194
4,"[7454, 2402, 257, 640, 11, 612, 373, 257, 1310...",160


In [15]:
train_tokenized_dataset['length']

Column([163, 178, 213, 194, 160])

In [16]:
len(train_tokenized_dataset['tokens'])

2119719

In [17]:
print("Total number of tokens in train set:", np.sum(train_tokenized_dataset["length"]))
print("Total number of tokens in val set:", np.sum(val_tokenized_dataset["length"]))

Total number of tokens in train set: 473992236
Total number of tokens in val set: 4765918


In [18]:
enc.decode(train_tokenized_dataset[0]['tokens'])

'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.<|endoftext|>'

In [19]:
info = np.iinfo(np.uint16)

print(f"Minimum value of uint16: {info.min}")
print(f"Maximum value of uint16: {info.max}")

Minimum value of uint16: 0
Maximum value of uint16: 65535


In [20]:
def save_to_binary(dataset, folder_path, split_name):
    arr_len = np.sum(dataset["length"], dtype=np.uint64)
    os.makedirs(folder_path, exist_ok=True)
    filename = f"data/{split_name}_{arr_len}tkns.bin"

    dtype = np.uint16
    print(f"Writing {filename} ({arr_len / 1e6:.2f}M tokens)...")
    arr = np.memmap(filename, dtype=dtype, mode="w+", shape=(arr_len,))

    idx = 0
    total_batches = 1024
    for batch_idx in tqdm(range(total_batches), desc=f"Writing {filename}"):
        batch = dataset.shard(
            num_shards=total_batches, index=batch_idx, contiguous=True
        ).with_format("numpy")

        arr_batch = np.concatenate(batch["tokens"]).astype(dtype)

        arr[idx : idx + len(arr_batch)] = arr_batch  # noqa: E203
        idx += len(arr_batch)

    arr.flush()
    print(f"Saved {filename}")

In [21]:
save_to_binary(train_tokenized_dataset, "data", "train")

Writing data/train_473992236tkns.bin (473.99M tokens)...


Writing data/train_473992236tkns.bin: 100%|██████████| 1024/1024 [03:31<00:00,  4.83it/s]

Saved data/train_473992236tkns.bin


In [22]:
save_to_binary(val_tokenized_dataset, "data", "val")

Writing data/val_4765918tkns.bin (4.77M tokens)...


Writing data/val_4765918tkns.bin: 100%|██████████| 1024/1024 [00:02<00:00, 355.23it/s]

Saved data/val_4765918tkns.bin
